# Practical demonstration 1: your turn

*SSTA workshop, 26 November 2026. Goes with Session 2, "ADB's approach to super-resolution".*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adb-sr/sr-client/blob/main/notebooks/demo1_your_turn.ipynb)

You have just seen the service at work on one example. Now you make a super-resolved image
of **a place you know** and judge it yourself. Work in pairs: one person runs the cells, the
other looks closely and answers the questions.

**About 20 minutes, five steps, four questions.** You do not need to write code:

* **run each grey cell once**, from top to bottom, with **Shift + Enter** (or the ▶ button);
* then use the **menus, buttons and sliders that appear under the cell**. They act at once,
  and your answers are saved as you type. You never need to run a cell twice.

| Step | What you do | Question |
|---|---|---|
| 1 | Pick a place and send it to the service | |
| 2 | Look at the 8 Sentinel-2 images the model uses | **Q1** How many are really clear? |
| 3 | Compare Stage 1 with Sentinel-2 | **Q2** What can you see now that you could not see before? |
| 4 | Look at Stage 2, the full result | **Q3** One detail it got right, one it got wrong |
| 5 | Turn one dial and compare | **Q4** Which version would you use, and for what? |

At the end, the notebook shows a **result card** with your answers. We compare the cards in
the room.

**Stuck?** Raise your hand, or carry on with *Cebu City and Mandaue*: every step works with
it.

## 0. Setup

Run the next three cells. The first takes about a minute.

In [ ]:
%pip install -q "sr-client[geo,viz] @ git+https://github.com/adb-sr/sr-client@v0.1.0"

Your API key starts with `srk_`. The cell asks for the key and the service URL (both are on
your card). In Colab you can also store them under **Secrets** (the key icon on the left) as
`SR_API_KEY` and `SR_API_URL`.

In [ ]:
import getpass
import os

from sr_client import SRClient, SRError, area_around


def secret(name, prompt):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return getpass.getpass(prompt) if "KEY" in name else input(prompt)


os.environ["SR_API_KEY"] = secret("SR_API_KEY", "API key (srk_...): ")
os.environ["SR_API_URL"] = secret("SR_API_URL", "Service URL: ")

sr = SRClient()
me = sr.me()
print(f"Key {me['prefix']} ({me['owner']}): ${me['budget']['remaining_usd']:.2f} of budget left")

In [ ]:
# @title Helper functions (run this cell; you do not need to read it) { display-mode: "form" }
import threading
import time
from datetime import date

import httpx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rioxarray
import xarray as xr
from rasterio.enums import Resampling
from rasterio.warp import transform, transform_bounds
from rasterio.windows import from_bounds
from scipy.ndimage import binary_erosion

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
EARTH_SEARCH = "https://earth-search.aws.element84.com/v1/collections/sentinel-2-l2a/items/"


def manifest(job):
    """The job's manifest.json: scenes used per tile, parameters, cost, warnings."""
    return httpx.get(job.results()["files"]["manifest.json"], timeout=60).json()


def wait_for_stage(job, stage, poll_s=10):
    """Wait until a stage's result is published (stage 1 is published before stage 2)."""
    t0 = time.monotonic()
    while True:
        job.refresh()
        if job.status in ("failed", "cancelled"):
            raise RuntimeError(f"job {job.id} {job.status}: {job.info.get('error')}")
        try:
            if stage in job.results()["stages_done"]:
                print(f"\rstage {stage} ready after {(time.monotonic() - t0) / 60:.1f} min".ljust(60))
                return job
        except SRError as exc:
            if exc.status != 409:  # 409 = no result yet
                raise
        print(f"\r{job.status}, {(time.monotonic() - t0) / 60:.1f} min ...", end="", flush=True)
        time.sleep(poll_s)


def load(job, name):
    """A result file as an xarray DataArray in reflectance (0-1), read into memory."""
    return job.open(name).astype("float32").load()


def clip(da, lat, lon, km):
    """The km x km square around a point (works for 2.5 m and 10 m images alike)."""
    x0, y0, x1, y1 = transform_bounds("EPSG:4326", da.rio.crs, *area_around(lat, lon, km))
    return da.rio.clip_box(x0, y0, x1, y1)


def rgb(da, bands=(1, 2, 3), ceiling=0.3):
    """Natural colour (or any 3 bands) scaled for display: reflectance / ceiling."""
    a = np.moveaxis(da.sel(band=list(bands)).values, 0, -1)
    return np.clip(np.nan_to_num(a) / ceiling, 0, 1)


def panel_fig(images, titles, size=4.2, bands=(1, 2, 3), ceiling=0.3, suptitle=None):
    """Images side by side as a matplotlib figure (each a DataArray or a ready RGB array)."""
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size + 0.4),
                             squeeze=False)
    for ax, img, title in zip(axes[0], images, titles):
        ax.imshow(rgb(img, bands, ceiling) if hasattr(img, "rio") else img,
                  interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, y=1.0)
    fig.tight_layout()
    return fig


def show(images, titles, **kw):
    panel_fig(images, titles, **kw)
    plt.show()


def tile_grid(m):
    """The job's tiles as (rows, cols, size): corner positions in 10 m pixels and the tile
    size (32 pixels = 320 m). Neighbouring tiles overlap."""
    rows = sorted({t["row"] for t in m["tiles"]})
    cols = sorted({t["col"] for t in m["tiles"]})
    return rows, cols, 32


def show_tiles(m, image, highlight=(3, 3)):
    """Left: the area with the outline of every tile, and two neighbours highlighted to show
    the overlap. Right: how many different Sentinel-2 images each tile used."""
    rows, cols, size = tile_grid(m)
    scale = image.sizes["x"] / m["grid"]["width"]          # image pixels per 10 m pixel
    fig, axes = plt.subplots(1, 2, figsize=(13, 6.2))
    ax = axes[0]
    ax.imshow(rgb(image), interpolation="nearest")
    for t in m["tiles"]:
        ax.add_patch(plt.Rectangle((t["col"] * scale, t["row"] * scale), size * scale,
                                   size * scale, fill=False, ec="white", lw=0.4, alpha=0.6))
    r, c = rows[min(highlight[0], len(rows) - 1)], cols[min(highlight[1], len(cols) - 2)]
    for col, colour in ((c, "#ffd400"), (cols[cols.index(c) + 1], "#00e5ff")):
        ax.add_patch(plt.Rectangle((col * scale, r * scale), size * scale, size * scale,
                                   fill=False, ec=colour, lw=2.5))
    step = (cols[1] - cols[0]) * 10 if len(cols) > 1 else size * 10
    ax.set_title(f"{len(m['tiles'])} tiles of {size * 10} m, one every {step} m: "
                 f"neighbours overlap by {size * 10 - step} m")
    ax.axis("off")
    count = np.full((len(rows), len(cols)), np.nan)
    for t in m["tiles"]:
        count[rows.index(t["row"]), cols.index(t["col"])] = len(set(t["scene_ids"]))
    im = axes[1].imshow(count, cmap="YlGnBu", vmin=0, vmax=8)
    for (i, j), v in np.ndenumerate(count):
        if np.isfinite(v):
            axes[1].text(j, i, int(v), ha="center", va="center", fontsize=9,
                         color="white" if v > 4 else "black")
    axes[1].set_title("different Sentinel-2 images used by each tile (of 8)")
    axes[1].axis("off")
    fig.colorbar(im, ax=axes[1], shrink=0.7)
    fig.tight_layout()
    plt.show()


def to_10m(da):
    """Average 4 x 4 pixels of a 2.5 m image: what Sentinel-2 would have seen."""
    return da.coarsen(x=4, y=4, boundary="trim").mean()


def enlarge(lr_da, like):
    """A 10 m image enlarged (bicubic) to the 2.5 m grid of ``like``."""
    return lr_da.rio.write_nodata(np.nan).rio.reproject_match(like, resampling=Resampling.cubic)


def agreement(sr_da, lr_da):
    """Correlation (0-1) between a 2.5 m image averaged to 10 m and the Sentinel-2 input,
    averaged over the 4 bands. 1 = the same pattern of light and dark."""
    a = to_10m(sr_da).values
    b = lr_da.values[:, :a.shape[1], :a.shape[2]]
    ok = np.isfinite(a).all(0) & np.isfinite(b).all(0)
    return float(np.mean([np.corrcoef(a[i][ok], b[i][ok])[0, 1] for i in range(4)]))


def detail(da):
    """How much fine detail an image has: the average brightness change from one pixel to
    the next (natural-colour bands)."""
    v = np.nan_to_num(da.sel(band=[1, 2, 3]).values.mean(0))
    return float(np.mean(np.abs(np.diff(v, axis=0))) + np.mean(np.abs(np.diff(v, axis=1))))


def scene_images(scene_ids, lat, lon, km):
    """Natural colour and cloud mask of each Sentinel-2 scene around a point, read straight
    from the public archive (Element 84 Earth Search, AWS)."""
    out = []
    for sid in scene_ids:
        item = httpx.get(EARTH_SEARCH + sid, timeout=60).json()
        with rasterio.open(item["assets"]["visual"]["href"]) as r:
            win = from_bounds(*transform_bounds("EPSG:4326", r.crs, *area_around(lat, lon, km)),
                              r.transform)
            img = np.moveaxis(r.read(window=win), 0, -1)
        with rasterio.open(item["assets"]["scl"]["href"]) as r:  # scene classification, 20 m
            win = from_bounds(*transform_bounds("EPSG:4326", r.crs, *area_around(lat, lon, km)),
                              r.transform)
            scl = r.read(1, window=win, out_shape=img.shape[:2], resampling=Resampling.nearest)
        # masked like the service does: everything but vegetation, bare soil, water and
        # "unclassified" (ESA scene classes 4-7), i.e. clouds, cirrus, shadows, no data
        masked = ~np.isin(scl, (4, 5, 6, 7))
        out.append((sid, img, masked))
    return out


def cloud_over_area(scene_ids, area):
    """For each Sentinel-2 scene, the share (0-1) of ``area`` (bbox W, S, E, N) that is cloud,
    cirrus or shadow in ESA's scene classification (the rule the service uses), among the
    pixels the scene covers. NaN if the scene does not cover the area. Read from the public
    archive, several scenes at a time."""
    from concurrent.futures import ThreadPoolExecutor

    def one(sid):
        item = httpx.get(EARTH_SEARCH + sid, timeout=60).json()
        with rasterio.open(item["assets"]["scl"]["href"]) as r:
            win = from_bounds(*transform_bounds("EPSG:4326", r.crs, *area), r.transform)
            scl = r.read(1, window=win, boundless=True, fill_value=0)
        covered = scl != 0
        if not covered.any():
            return float("nan")
        return float((~np.isin(scl, (4, 5, 6, 7)))[covered].mean())

    with ThreadPoolExecutor(8) as pool:
        return dict(zip(scene_ids, pool.map(one, scene_ids)))


def ndvi(da):
    """Vegetation index: (NIR - red) / (NIR + red)."""
    red, nir = da.sel(band=1), da.sel(band=4)
    return (nir - red) / (nir + red)


# ---- section 9: checking against real high-resolution images ----------------------------
SATELLOGIC = "https://satellogic-earthview.s3.us-west-2.amazonaws.com/data/tif"


def satellogic_scene(scene):
    """Date and centre (lat, lon) of a Satellogic EarthView scene, from its name."""
    d, _, _, zone, e, n = scene.split("_")
    epsg = (32600 if zone.endswith("N") else 32700) + int(zone[:-1])
    lon, lat = transform(f"EPSG:{epsg}", "EPSG:4326", [int(e) + 192], [int(n) - 192])
    return f"{d[:4]}-{d[4:6]}-{d[6:]}", lat[0], lon[0]


def load_satellogic(scene):
    """A Satellogic EarthView scene: 384 m x 384 m at 1 m; red, green, blue, NIR;
    top-of-atmosphere reflectance (0-1). Read from the public archive on AWS."""
    d, _, _, zone, e, n = scene.split("_")
    url = (f"{SATELLOGIC}/zone={zone}/region={e}_{n}/date={d[:4]}-{d[4:6]}-{d[6:]}/"
           f"{scene}_TOA.tif")
    da = rioxarray.open_rasterio(url).astype("float32").load()
    return da.where(da > 0) / 10000


def sentinel2_image(scene_id, like):
    """One Sentinel-2 image (red, green, blue, NIR, reflectance 0-1) on the 10 m grid of
    ``like``; clouds, shadows and no data are NaN."""
    item = httpx.get(EARTH_SEARCH + scene_id, timeout=60).json()
    props = item["properties"]
    offset = 0 if props.get("earthsearch:boa_offset_applied", True) else (
        -1000 if props.get("s2:processing_baseline", "00") >= "04.00" else 0)
    bands = []
    for name in ("red", "green", "blue", "nir", "scl"):
        da = rioxarray.open_rasterio(item["assets"][name]["href"])
        x0, y0, x1, y1 = like.rio.transform_bounds(da.rio.crs)
        da = da.rio.clip_box(x0 - 50, y0 - 50, x1 + 50, y1 + 50)
        bands.append(da.rio.reproject_match(like, resampling=Resampling.nearest)[0]
                     .astype("float32"))
    clear = np.isin(bands[4], (4, 5, 6, 7)) & (bands[0] > 0)
    out = xr.concat([(b + offset) / 10000 for b in bands[:4]], dim="band")
    out = out.assign_coords(band=[1, 2, 3, 4]).where(clear)
    return out.rio.write_crs(like.rio.crs).rio.write_transform(like.rio.transform())


def _corr(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    return float(np.corrcoef(a[ok], b[ok])[0, 1]) if ok.sum() > 50 else np.nan


def _moved(da, dx, dy):
    """The same image moved dx metres east and dy metres north."""
    out = da.assign_coords(x=da.x + dx, y=da.y + dy)
    return out.rio.write_transform(out.rio.transform(recalc=True))


def _averaged_onto(vhr, like):
    """The 1 m image averaged onto the grid of ``like`` (10 m or 2.5 m)."""
    return vhr.rio.write_nodata(np.nan).rio.reproject_match(like, resampling=Resampling.average)


def _align(vhr, s2):
    """Shift (metres east, north) that best lines the 1 m image up with Sentinel-2 at 10 m:
    a search in 3 m steps, then 1 m steps around the best."""
    def fit(dx, dy):
        a = _averaged_onto(_moved(vhr, dx, dy), s2).values
        return np.nanmean([_corr(a[i], s2.values[i]) for i in range(4)])

    best = max(((dx, dy) for dx in range(-24, 25, 3) for dy in range(-24, 25, 3)),
               key=lambda s: fit(*s))
    return max(((best[0] + i, best[1] + j) for i in range(-2, 3) for j in range(-2, 3)),
               key=lambda s: fit(*s))


def _match_brightness(vhr, s2):
    """Fit each band of the 1 m image to Sentinel-2 at 10 m (a straight line per band)."""
    a = _averaged_onto(vhr, s2).values
    out = vhr.copy()
    for i in range(4):
        ok = np.isfinite(a[i]) & np.isfinite(s2.values[i])
        gain, offset = np.polyfit(a[i][ok], s2.values[i][ok], 1)
        out[i] = vhr[i] * gain + offset
    return out


def _nudge(product, ref):
    """Move a product by up to one 2.5 m pixel to where it best matches the reference:
    two satellites never line up to the metre."""
    def fit(s):
        q = np.roll(product.values, s, axis=(-2, -1))
        return np.nanmean([_corr(q[b], ref.values[b]) for b in range(4)])

    s = max(((i, j) for i in (-1, 0, 1) for j in (-1, 0, 1)), key=fit)
    q = np.roll(product.values, s, axis=(-2, -1))
    if s[0]:
        q[..., 0 if s[0] > 0 else -1, :] = np.nan   # the row that wrapped around
    if s[1]:
        q[..., :, 0 if s[1] > 0 else -1] = np.nan
    return product.copy(data=q)


def fine_part(x):
    """A 2.5 m image minus its own 10 m average: the detail smaller than a Sentinel-2 pixel."""
    h, w = (x.shape[-2] // 4) * 4, (x.shape[-1] // 4) * 4
    x = x[..., :h, :w]
    m = np.nanmean(x.reshape(*x.shape[:-2], h // 4, 4, w // 4, 4), axis=(-3, -1))
    return x - np.repeat(np.repeat(m, 4, -2), 4, -1)


def compare_with_vhr(job, scene):
    """Sentinel-2, Stage 1 and Stage 2 next to a real 1 m image of the same place, all on the
    2.5 m grid of the job and cut to the 1 m image's footprint."""
    day = satellogic_scene(scene)[0]
    lr, s1, s2 = (load(job, f) for f in ("lr_reference.tif", "stage1_sr.tif", "stage2_sr.tif"))
    sid, info = min(manifest(job)["scenes"].items(), key=lambda kv: abs(
        (date.fromisoformat(kv[1]["datetime"][:10]) - date.fromisoformat(day)).days))
    s2_day = sentinel2_image(sid, lr)
    vhr = load_satellogic(scene)
    dx, dy = _align(vhr, s2_day)
    ref = _averaged_onto(_match_brightness(_moved(vhr, dx, dy), s2_day), s1)
    foot = binary_erosion(np.isfinite(ref.values).all(0), iterations=1)
    ys, xs = np.where(foot)
    # start on a 10 m pixel (4 x 4 output pixels), so fine_part's blocks are Sentinel-2's
    cut = dict(y=slice(ys.min() // 4 * 4, ys.max() + 1), x=slice(xs.min() // 4 * 4, xs.max() + 1))
    ref = ref.where(foot).isel(**cut)
    s2_date = info["datetime"][:10]
    gap = (date.fromisoformat(s2_date) - date.fromisoformat(day)).days
    images = {f"Sentinel-2 ({'same day' if gap == 0 else f'{gap:+d} days'}), enlarged":
              enlarge(s2_day, s1), "Stage 1": s1, "Stage 2": s2}
    images = {k: _nudge(v.where(foot).isel(**cut), ref) for k, v in images.items()}
    images["Satellogic, 1 m averaged to 2.5 m"] = ref
    print(f"Satellogic {day}; Sentinel-2 {s2_date}; Satellogic image moved {dx:+d} m east, "
          f"{dy:+d} m north to line up with Sentinel-2")
    return images


def show_vhr(images, title):
    fig, axes = plt.subplots(3, 4, figsize=(15, 12.5), layout="constrained")
    for j, (name, da) in enumerate(images.items()):
        axes[0, j].imshow(rgb(da), interpolation="nearest")
        axes[0, j].set_title(name, fontsize=10)
        axes[1, j].imshow(da.sel(band=4).values, cmap="gray", vmin=0, vmax=0.5,
                          interpolation="nearest")
        im = axes[2, j].imshow(ndvi(da).values, cmap="RdYlGn", vmin=-0.2, vmax=0.8,
                               interpolation="nearest")
    for ax, label in zip(axes[:, 0], ["natural colour", "near-infrared", "NDVI"]):
        ax.text(-0.04, 0.5, label, transform=ax.transAxes, rotation=90, ha="right",
                va="center", fontsize=11)
    for ax in axes.ravel():
        ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=axes[2, :], orientation="horizontal", shrink=0.4, label="NDVI")
    fig.suptitle(title)
    plt.show()


def vhr_scores(images):
    """Correlation with the real image: all detail, and the detail smaller than 10 m."""
    ref = images["Satellogic, 1 m averaged to 2.5 m"]
    rows = {}
    for name, da in images.items():
        if da is ref:
            continue
        row = {}
        for label, idx in (("colour", [0, 1, 2]), ("NIR", [3])):
            row[f"{label}: all"] = np.mean([_corr(da.values[i], ref.values[i]) for i in idx])
            row[f"{label}: < 10 m"] = np.mean([_corr(fine_part(da.values[i]),
                                                     fine_part(ref.values[i])) for i in idx])
        a, b = ndvi(da).values, ndvi(ref).values
        row["NDVI: all"], row["NDVI: < 10 m"] = _corr(a, b), _corr(fine_part(a), fine_part(b))
        rows[name.split(" (")[0]] = row
    return pd.DataFrame(rows).T


class Sweep:
    """Submits several jobs one after another in the background, waiting whenever the key's
    queue is full, so the rest of the notebook can go on meanwhile.
    ``settings``: {name: (area, date, {parameters})}."""

    def __init__(self, settings):
        self.settings, self.jobs, self.errors = settings, {}, {}
        client = SRClient()  # its own connection, used only by the background thread

        def run():
            for name, (area, target_date, params) in settings.items():
                while True:
                    try:
                        self.jobs[name] = client.submit(area, target_date, **params)
                        break
                    except SRError as exc:
                        if exc.status == 429:  # queue full: try again shortly
                            time.sleep(15)
                            continue
                        self.errors[name] = exc.message
                        break

        self.thread = threading.Thread(target=run, daemon=True)
        self.thread.start()

    def status(self):
        for name in self.settings:
            job = self.jobs.get(name)
            state = (self.errors.get(name) or ("waiting to submit" if job is None
                                               else job.refresh().status))
            print(f"  {name:<22} {state}")

    def wait(self):
        while self.thread.is_alive() or any(
                j.refresh().status not in ("done", "partial", "failed", "cancelled")
                for j in self.jobs.values()):
            done = sum(j.status in ("done", "partial") for j in self.jobs.values())
            print(f"\r{done}/{len(self.settings)} done ...", end="", flush=True)
            time.sleep(10)
        print(f"\r{sum(j.status in ('done', 'partial') for j in self.jobs.values())}/"
              f"{len(self.settings)} done".ljust(40))
        for name, msg in self.errors.items():
            print(f"  {name}: not submitted: {msg}")
        return self.jobs


# ---- the worksheet ("your turn") ---------------------------------------------------------

def view(da, lat, lon, size_m=500, east_m=0, north_m=0):
    """A square of size_m metres; its centre is moved east_m and north_m from (lat, lon)."""
    lat2 = lat + north_m / 111_320
    lon2 = lon + east_m / (111_320 * np.cos(np.radians(lat)))
    return clip(da, lat2, lon2, size_m / 1000)


def show_inputs(job, lat, lon, target_date, km=0.5):
    """The 8 Sentinel-2 images the model used for the tile in the middle of the area, with
    the masked pixels (cloud, cirrus, shadow, no data) outlined in red."""
    m = manifest(job)
    target = date.fromisoformat(str(target_date))
    mid_r, mid_c = m["grid"]["height"] / 2, m["grid"]["width"] / 2
    tile = min(m["tiles"],
               key=lambda t: (t["row"] + 16 - mid_r) ** 2 + (t["col"] + 16 - mid_c) ** 2)
    scenes = scene_images(tile["scene_ids"], lat, lon, km)
    fig, axes = plt.subplots(1, len(scenes), figsize=(2.3 * len(scenes), 2.9), squeeze=False)
    for i, (ax, (sid, img, masked), d) in enumerate(zip(axes[0], scenes, tile["dates"]), 1):
        ax.imshow(img)
        if masked.any() and not masked.all():
            ax.contour(masked, levels=[0.5], colors="red", linewidths=0.8)
        days = (date.fromisoformat(d) - target).days
        note = (" (reference)" if i == 1
                else " (repeat)" if sid in tile["scene_ids"][:i - 1] else "")
        ax.set_title(f"{i}{note}\n{d}, {days:+d} days\n{masked.mean():.0%} masked", fontsize=8)
        ax.axis("off")
    fig.suptitle(f"The 8 images behind the middle of your area ({int(km * 1000)} m across)",
                 fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()
    distinct = len(set(tile["scene_ids"]))
    print(f"{len(m['scenes'])} different images used over the whole area; "
          f"this tile uses {distinct} different ones" +
          ("" if distinct == len(tile["scene_ids"]) else " (some twice)"))
    return tile


WARNING_MEANING = {
    "water": "detail in shallow water and on reefs is not reliable; use Sentinel-2 there",
    "low_diversity": "some tiles had fewer than 4 different clear images: less real detail",
    "nodata": "parts of the area have no clear image at all and are empty",
    "haze": "some input images are hazy: colours may be washed out",
    "missing_tiles": "some tiles could not be computed",
}


def seed_difference(a, b):
    """Mean absolute difference of two images over the natural-colour bands."""
    return np.abs(a.sel(band=[1, 2, 3]) - b.sel(band=[1, 2, 3])).mean("band")


PLACES = {'ADB headquarters, Mandaluyong': {'lat': 14.5878,
                                   'lon': 121.0585,
                                   'date': '2026-04-15',
                                   'note': 'dense city: office towers, malls, residential '
                                           'blocks'},
 'DHSUD office, Quezon City': {'lat': 14.6447,
                               'lon': 121.0514,
                               'date': '2026-04-15',
                               'note': 'city: government offices, tree-lined streets, housing'},
 'PhilSA office, Quezon City': {'lat': 14.6096,
                                'lon': 121.0803,
                                'date': '2026-04-15',
                                'note': 'city by the Marikina river: towers, subdivisions, '
                                        'riverside'},
 'Cebu City and Mandaue': {'lat': 10.33,
                           'lon': 123.92,
                           'date': '2026-03-20',
                           'note': 'dense city by the coast: port, warehouses, small houses'},
 'Nueva Ecija rice farms': {'lat': 15.5784,
                            'lon': 120.9726,
                            'date': '2026-03-15',
                            'note': 'rice fields, villages and farm-to-market roads'}}
EXPERIMENTS = {'less noise (tau 100)': {'tau': 100},
 'more noise (tau 500)': {'tau': 500, 'nfe': 250},
 'much more noise (tau 999)': {'tau': 999, 'nfe': 250},
 'fewer steps (10 steps)': {'tau': 250, 'nfe': 10},
 'another random seed (seed 1)': {'tau': 250, 'seed': 1}}


# ---- controls shared by the workshop notebooks: answers that save themselves, figures as images
import html as _html
import io

import ipywidgets as W
from IPython.display import display

ANSWERS = {}    # your answers; saved as soon as you type or click
UI = {}         # the controls of each step (used for testing)
_WIDE = W.Layout(width="520px")
_LABEL = {"description_width": "150px"}


def _png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()


def _note(text, colour="#333"):
    return f"<div style='color:{colour}; line-height:1.5'>{text}</div>"


def _pre(text):
    return f"<pre style='margin:4px 0; line-height:1.35'>{_html.escape(text)}</pre>"


# ---- answers that save themselves ------------------------------------------------------------

def ask(*questions):
    """Each question is (key, text, kind): kind is "text" or a list of choices. The answer
    is saved in ANSWERS the moment it changes; nothing to run."""
    rows = []
    for key, text, kind in questions:
        saved = W.HTML()
        if kind == "text":
            field = W.Text(value=ANSWERS.get(key, ""), placeholder="type here",
                           continuous_update=True, layout=_WIDE)
        else:
            field = W.ToggleButtons(options=list(kind), value=ANSWERS.get(key),
                                    style={"button_width": "auto"})

        def on_change(change, key=key, saved=saved):
            ANSWERS[key] = change["new"]
            saved.value = _note("saved ✓", "#1a7f37") if change["new"] not in ("", None) else ""
        field.observe(on_change, names="value")
        rows.append(W.VBox([W.HTML(f"<b>{text}</b>"), W.HBox([field, saved])],
                           layout=W.Layout(margin="6px 0 10px 0")))
        rows[-1].field = field
    box = W.VBox(rows, layout=W.Layout(border="1px solid #007DB7", padding="8px 12px",
                                       margin="8px 0"))
    box.fields = [r.field for r in rows]
    display(box)
    return box


# ---- the worksheet's controls: dropdowns, buttons and sliders that act at once ---------------
S = {}          # this session: the place, the jobs and the images that are already loaded
OWN_PLACE = "My own place"
_DONE = ("done", "partial", "failed", "cancelled")


def _ready(*keys):
    """True if the earlier steps were done; otherwise says which step is missing."""
    if "job" not in S:
        display(W.HTML(_note("<b>Pick a place in step 1 first</b>: choose it and press the "
                             "button, then run this cell again.", "#b00020")))
        return False
    return True


def _wait(job, status=None, until_stage=None):
    """Poll a job until it is finished (or until a stage is published), showing progress."""
    t0 = time.monotonic()
    while True:
        job.refresh()
        steps = " · ".join(f"{p['step']} {p['state']}" for p in (job.info.get("progress") or [])
                           if p["state"] != "waiting")
        line = f"{job.status}, {(time.monotonic() - t0) / 60:.1f} min   {steps}"
        if status is not None:
            status.value = _note(line)
        if job.status in _DONE:
            return job
        if until_stage:
            try:
                if until_stage in job.results()["stages_done"]:
                    return job
            except SRError as exc:
                if exc.status != 409:   # 409 = no result yet
                    raise
        time.sleep(5)


def _need(name):
    """Load an image of the main job once and keep it."""
    files = {"lr": "lr_reference.tif", "s1": "stage1_sr.tif", "s2": "stage2_sr.tif"}
    if name not in S:
        S[name] = load(S["job"], files[name])
    return S[name]


# ---- step 1: pick a place -----------------------------------------------------------------

def pick_place(sr):
    """Choose a place and send it to the service with one button."""
    place = W.Dropdown(options=[*PLACES, OWN_PLACE], value="Cebu City and Mandaue",
                       description="Place", layout=_WIDE, style=_LABEL)
    lat = W.FloatText(value=14.6760, description="Latitude", style=_LABEL)
    lon = W.FloatText(value=121.0437, description="Longitude", style=_LABEL)
    day = W.Text(value="2026-04-15", description="Date (YYYY-MM-DD)", style=_LABEL)
    own = W.VBox([W.HTML(_note("A point (in Google Maps: right-click, then click the "
                               "coordinates to copy them) and a date in the dry season.")),
                  lat, lon, day])
    own.layout.display = "none"
    about = W.HTML()
    send = W.Button(description="Send to the service", button_style="primary", icon="rocket",
                    layout=W.Layout(width="220px"))
    status = W.HTML()

    def describe(_=None):
        own.layout.display = "" if place.value == OWN_PLACE else "none"
        about.value = _note("" if place.value == OWN_PLACE else PLACES[place.value]["note"],
                            "#555")
    place.observe(describe, names="value")
    describe()

    def on_send(_):
        send.disabled = True
        try:
            if place.value == OWN_PLACE:
                la, lo, d = float(lat.value), float(lon.value), day.value.strip()
                date.fromisoformat(d)
            else:
                p = PLACES[place.value]
                la, lo, d = p["lat"], p["lon"], p["date"]
            status.value = _note("Asking the service which Sentinel-2 images exist ...")
            area = area_around(la, lo, km=2)
            est = sr.estimate(area, d)
            lines = [f"{place.value}, {d}",
                     f"{len(est['scenes'])} Sentinel-2 images found between "
                     f"{est['search_window'][0]} and {est['search_window'][1]}",
                     f"{est['tiles']} tiles of 320 m; cost if nothing was computed before: "
                     f"${est['estimated_usd']:.2f}"]
            lines += [f"warning: {w['message']}" for w in est["warnings_likely"]]
            if est["problems"]:
                lines += [f"cannot send this job: {p}" for p in est["problems"]]
                status.value = _pre("\n".join(lines)) + _note(
                    "Choose another place or date and press the button again.", "#b00020")
                return
            job = sr.submit(area, d)
            S.clear()
            S.update(sr=sr, place=place.value, lat=la, lon=lo, date=d, area=area, est=est,
                     job=job)
            ANSWERS["PLACE"] = f"{place.value} ({d})"
            status.value = _pre("\n".join(lines)) + _note(
                f"<b>Job sent</b> ({job.id[:8]}, {job.status}). Go on with step 2.", "#1a7f37")
        except ValueError:
            status.value = _note("The date must look like 2026-04-15.", "#b00020")
        except SRError as exc:
            status.value = _note(f"The service said: {_html.escape(exc.message)}", "#b00020")
        finally:
            send.disabled = False
    send.on_click(on_send)

    ui = W.VBox([place, about, own, send, status])
    ui.place, ui.send, ui.status, ui.own_fields = place, send, status, (lat, lon, day)
    UI["place"] = ui
    display(ui)


# ---- a viewer: move and zoom, the picture follows -------------------------------------------

class Viewer:
    """Images side by side with sliders to move the view and buttons to zoom. ``layers`` is
    a list of (title, image) or (title, image, "difference")."""

    def __init__(self, layers, size_m=500, panel=5.2, footer=None):
        self.layers, self.panel, self.footer = layers, panel, footer
        self.east = W.IntSlider(value=0, min=-750, max=750, step=50, continuous_update=False,
                                description="← west | east →", layout=_WIDE, style=_LABEL)
        self.north = W.IntSlider(value=0, min=-750, max=750, step=50, continuous_update=False,
                                 description="← south | north →", layout=_WIDE, style=_LABEL)
        self.size = W.ToggleButtons(options=[("250 m", 250), ("500 m", 500), ("1 km", 1000),
                                             ("whole area", 2000)], value=size_m,
                                    description="Zoom", style={"button_width": "90px",
                                                               "description_width": "150px"})
        self.image = W.Image(format="png")
        for w in (self.east, self.north, self.size):
            w.observe(self.draw, names="value")
        self.box = W.VBox([self.size, self.east, self.north, self.image])
        self.draw()

    def draw(self, _=None):
        size, east, north = self.size.value, self.east.value, self.north.value
        lat, lon = S["lat"], S["lon"]
        n = len(self.layers)
        fig, axes = plt.subplots(1, n + 1, figsize=(self.panel * n + 2.6, self.panel + 0.5),
                                 gridspec_kw={"width_ratios": [2.6] + [self.panel] * n})
        # where am I: the whole area with a frame around the current view
        whole = self.layers[-1 if len(self.layers[-1]) == 2 else 0][1]
        thumb = whole.isel(x=slice(None, None, 4), y=slice(None, None, 4))
        x0, y0 = float(whole.x.mean()), float(whole.y.mean())
        ext = [float(whole.x.min()), float(whole.x.max()), float(whole.y.min()),
               float(whole.y.max())]
        axes[0].imshow(rgb(thumb), extent=ext, interpolation="nearest")
        axes[0].add_patch(plt.Rectangle((x0 + east - size / 2, y0 + north - size / 2), size,
                                        size, fill=False, ec="#ffd400", lw=2))
        axes[0].set_title("where you are", fontsize=9)
        for ax, layer in zip(axes[1:], self.layers):
            cut = view(layer[1], lat, lon, size, east, north)
            if len(layer) == 3:     # a difference map
                ax.imshow(np.clip(cut.values / 0.05, 0, 1), cmap="magma", vmin=0, vmax=1,
                          interpolation="nearest")
            else:
                ax.imshow(rgb(cut), interpolation="nearest")
            ax.set_title(layer[0])
        for ax in axes:
            ax.set_xticks([])
            ax.set_yticks([])
        fig.suptitle(f"{S['place']}: {size} m across" + (f"   {self.footer}" if self.footer
                                                         else ""), y=1.0, fontsize=10)
        fig.tight_layout()
        self.image.value = _png(fig)


# ---- steps 2 to 4 -------------------------------------------------------------------------

def step2_inputs():
    """The 8 Sentinel-2 images behind the middle of the area, and question 1."""
    if not _ready():
        return
    wait_for_stage(S["job"], "1")
    S["tile"] = show_inputs(S["job"], S["lat"], S["lon"], S["date"])
    UI["q1"] = ask(("Q1_CLEAR_IMAGES", "Q1. How many of the 8 images are really clear over your "
                "place?", [str(i) for i in range(9)]),
               ("Q1_MISSED_CLOUD", "Is there cloud, haze or shadow that the red outline "
                "missed?", ["no", "yes", "not sure"]))


def step3_stage1():
    """Sentinel-2 next to Stage 1, and question 2."""
    if not _ready():
        return
    wait_for_stage(S["job"], "1")
    viewer = Viewer([("Sentinel-2, 10 m", _need("lr")), ("Stage 1, 2.5 m", _need("s1"))],
                    panel=6.2)
    display(viewer.box)
    UI["view3"] = viewer
    UI["q2"] = ask(("Q2_NEW_IN_STAGE_1", "Q2. Name one thing you can make out in Stage 1 that you could "
         "not make out in Sentinel-2.", "text"))


def step4_stage2():
    """Sentinel-2, Stage 1 and Stage 2, and question 3."""
    if not _ready():
        return
    status = W.HTML()
    display(status)
    job = _wait(S["job"], status)
    if job.status not in ("done", "partial"):
        status.value = _note(f"The job {job.status}: {job.info.get('error')}", "#b00020")
        return
    status.value = _pre("\n".join(
        [f"warning ({w['code']}): {WARNING_MEANING.get(w['code'], w.get('message', ''))}"
         for w in job.warnings] or ["warnings: none"]))
    viewer = Viewer([("Sentinel-2, 10 m", _need("lr")), ("Stage 1", _need("s1")),
                     ("Stage 2", _need("s2"))])
    display(viewer.box)
    UI["view4"] = viewer
    UI["q3"] = ask(("Q3_RIGHT", "Q3. One detail Stage 2 got right:", "text"),
        ("Q3_WRONG", "One detail Stage 2 got wrong or invented:", "text"))


# ---- step 5: one experiment -----------------------------------------------------------------

def step5_experiment():
    """Choose one change to the dials, guess, press the button, compare."""
    if not _ready():
        return
    choice = W.Dropdown(options=list(EXPERIMENTS), value="much more noise (tau 999)",
                        description="My experiment", layout=_WIDE, style=_LABEL)
    guess = W.ToggleButtons(options=["sharper, and I would still trust it",
                                     "sharper, but I would not trust the detail",
                                     "less sharp", "hardly any difference"], value=None,
                            style={"button_width": "auto"})
    guess.observe(lambda c: ANSWERS.__setitem__("MY_GUESS", c["new"]), names="value")
    run = W.Button(description="Run my experiment", button_style="primary", icon="flask",
                   layout=W.Layout(width="220px"))
    status = W.HTML()
    result = W.VBox([])

    def on_run(_):
        run.disabled = True
        try:
            name = choice.value
            status.value = _note("Sending the job ...")
            trial = S["sr"].submit(S["area"], S["date"], **EXPERIMENTS[name])
            _wait(trial, status)
            if trial.status not in ("done", "partial"):
                status.value = _note(f"The job {trial.status}: {trial.info.get('error')}",
                                     "#b00020")
                return
            status.value = _note("Loading the image ...")
            s1, s2, lr = _need("s1"), _need("s2"), _need("lr")
            s2_trial = load(trial, "stage2_sr.tif")
            diff = seed_difference(s2, s2_trial)
            S.update(trial=trial, s2_trial=s2_trial, experiment=name)
            ANSWERS["EXPERIMENT"] = name
            rows = [("Stage 1", s1), ("default Stage 2", s2), (name, s2_trial)]
            table = (f"{'':<34}{'agreement with':>16}{'fine':>10}\n"
                     f"{'':<34}{'Sentinel-2':>16}{'detail':>10}\n" +
                     "\n".join(f"{label:<34}{agreement(da, lr):>16.2f}{detail(da):>10.4f}"
                               for label, da in rows) +
                     f"\n\npixels that differ clearly between the two versions: "
                     f"{float((diff > 0.02).mean()):.0%}")
            viewer = Viewer([("default (tau 250, 250 steps, seed 0)", s2), (name, s2_trial),
                             ("where they differ (bright = more)", diff, "difference")],
                            size_m=250)
            UI["view5"] = viewer
            result.children = (viewer.box, W.HTML(_pre(table)))
            status.value = _note("Done. Move the view to compare.", "#1a7f37")
        except SRError as exc:
            status.value = _note(f"The service said: {_html.escape(exc.message)}", "#b00020")
        finally:
            run.disabled = False
    run.on_click(on_run)

    ui = W.VBox([choice, W.HTML("<b>Before you run it: what do you expect?</b>"), guess, run,
                 status, result])
    ui.choice, ui.guess, ui.run, ui.status, ui.result = choice, guess, run, status, result
    UI["experiment"] = ui
    display(ui)


def step5_question():
    options = ["Sentinel-2", "Stage 1", "default Stage 2", "my experiment"]
    UI["q4"] = ask(("Q4_REPORT_PICTURE", "Q4. Which version would you use for a picture in a "
                "report?", options),
               ("Q4_COUNTING", "And which for counting buildings or measuring roads?", options))


# ---- step 6: the result card ----------------------------------------------------------------

def card_text():
    def get(key):
        v = ANSWERS.get(key)
        return "(not answered)" if v in ("", None) else v

    lines = ["=" * 66, f"PLACE       {get('PLACE')}"]
    if "s1" in S and "lr" in S:
        lines.append(f"AGREEMENT   with Sentinel-2 at 10 m: Stage 1 "
                     f"{agreement(S['s1'], S['lr']):.2f}" +
                     (f", Stage 2 {agreement(S['s2'], S['lr']):.2f}" if "s2" in S else ""))
    if "job" in S:
        codes = [w["code"] for w in S["job"].warnings]
        lines.append(f"WARNINGS    {', '.join(dict.fromkeys(codes)) if codes else 'none'}")
    lines += ["-" * 66,
              f"Q1  clear images out of 8:        {get('Q1_CLEAR_IMAGES')}",
              f"    a cloud the mask missed:      {get('Q1_MISSED_CLOUD')}",
              f"Q2  new in Stage 1:               {get('Q2_NEW_IN_STAGE_1')}",
              f"Q3  Stage 2 got right:            {get('Q3_RIGHT')}",
              f"    Stage 2 got wrong:            {get('Q3_WRONG')}",
              f"Q4  my experiment:                {get('EXPERIMENT')}",
              f"    my guess was:                 {get('MY_GUESS')}",
              f"    for a picture in a report:    {get('Q4_REPORT_PICTURE')}",
              f"    for counting buildings:       {get('Q4_COUNTING')}",
              "=" * 66]
    return "\n".join(lines)


def result_card():
    """Your answers on one card; the button brings it up to date."""
    card = W.HTML(_pre(card_text()))
    again = W.Button(description="Update the card", icon="refresh")
    again.on_click(lambda _: setattr(card, "value", _pre(card_text())))
    ui = W.VBox([card, again])
    ui.card, ui.again = card, again
    UI["card"] = ui
    display(ui)

## 1. Pick a place *(2 minutes)*

Run the cell, choose the place you know best, and press **Send to the service**. All five
were run before the workshop, so they come back quickly and cost nothing.

| Place | What is there |
|---|---|
| **ADB headquarters, Mandaluyong** | dense city: office towers, malls, residential blocks |
| **DHSUD office, Quezon City** | city: government offices, tree-lined streets, housing |
| **PhilSA office, Quezon City** | city by the Marikina river: towers, subdivisions, riverside |
| **Cebu City and Mandaue** | dense city by the coast: port, warehouses, small houses |
| **Nueva Ecija rice farms** | rice fields, villages and farm-to-market roads |

Each area is a 2 km square. The service first makes an **estimate**, which is free
(how many Sentinel-2 images exist around the date, and what the job would cost), and then
starts the job. Both stages run; Stage 1 is ready first.

*My own place* is the last entry in the menu: fill in a point and a date. It takes a few
minutes longer and uses your budget, so do the five steps with a listed place first if you
are short of time. If you change the place later, run the cells of steps 2 to 5 again.

In [ ]:
pick_place(sr)

## 2. What did the satellite see? *(4 minutes)*

For every 320 m square, the model gets the **8 clearest Sentinel-2 images** closest to your
date. The cell shows the 8 for the middle of your area, read from the public Sentinel-2
archive (this takes about half a minute).

* Pixels outlined in **red** are masked: ESA's classification marks them as cloud, cirrus or
  shadow. The model fills them from the other images.
* Image 1 is the **reference**: the result follows its layout and colours.
* The classification is not perfect. It misses thin cloud and haze, and sometimes masks
  bright roofs.

Answer **Q1** in the blue box under the images.

In [ ]:
step2_inputs()

## 3. Stage 1: eight images become one *(4 minutes)*

Stage 1 combines the 8 images into one image with 2.5 m pixels. It is sharper than
Sentinel-2 but still soft.

Use the **sliders** to move to a spot you know: a street you walk, your office, a market, a
field. The small picture on the left shows where you are in the area; the **Zoom** buttons
change how much you see. Then answer **Q2**.

In [ ]:
step3_stage1()

## 4. Stage 2: the model adds detail *(5 minutes)*

Stage 2 adds noise to the Stage 1 image and removes it step by step. While it removes the
noise, it draws small details the way it learned them from 1.8 million real high-resolution
images. The result looks sharp. Some of that detail is real, and some is only *plausible*.

You know this place, so you can tell the difference. Move the view again and look for:

* something Stage 2 shows **correctly** (a road that is really there, the shape of a building
  you know);
* something that is **wrong or invented** (a roof pattern that does not exist, a road that
  ends nowhere, a building that became trees).

Then answer **Q3**.

In [ ]:
step4_stage2()

## 5. One experiment: turn a dial *(5 minutes)*

Stage 2 has the two dials from Session 2:

* **Dial 1, `tau`**: how much noise is added, from 0 (Stage 1 only) to 999 (almost only
  noise). More noise lets the model draw more of its own detail. The default is 250. (The
  noise experiments keep the default 250 steps, so only one dial changes.)
* **Dial 2, `nfe`**: in how many steps the noise is removed. Fewer steps are faster and
  cheaper, and less sharp.
* The noise is random. The **seed** fixes it: another seed gives another version of the
  invented detail, while everything that was measured stays the same.

Choose **one** experiment in the menu; other pairs will choose others, and we compare at the
end. Click what you expect, then press **Run my experiment**. A setting that was run before
comes back at once; a new one takes about 4 minutes.

The result shows the default image, yours, and **where the two differ** (bright = large
difference). If your experiment was the other seed, that third picture is a map of where the
model invented detail. Under the pictures are two numbers for each version:

* *agreement with Sentinel-2*: averaged back to 10 m, does the image still match what the
  satellite measured (1 = same pattern)?
* *fine detail*: how much the brightness changes from one pixel to the next (higher =
  sharper).

In [ ]:
step5_experiment()

When you have compared the two versions, answer **Q4**.

In [ ]:
step5_question()

## 6. Your result card

Run the cell. If you change an answer afterwards, press **Update the card**. Enter your
answers in the poll (the link is on the slide), and be ready to say in one sentence what you
found at your place.

In [ ]:
result_card()

---
## If you have time

**Take the images home.** The files open in QGIS or ArcGIS. In Colab they appear in the file
browser on the left, under `sr_results/<job id>/`.

| File | What it holds |
|---|---|
| `stage1_sr.tif`, `stage2_sr.tif` | the 2.5 m images: red, green, blue, near-infrared; reflectance × 10,000 |
| `lr_reference.tif` | the Sentinel-2 input, 10 m |
| `validity.tif` | 0 no data · 1 valid · 2 low diversity (fewer than 4 different clear images) |
| `manifest.json` | the images used for every tile, the settings, model versions, cost |

In [ ]:
job = S["job"]                      # the job of the place you picked in step 1
paths = job.download("sr_results")
for name, path in paths.items():
    print(f"  {path}  ({path.stat().st_size / 1e6:.1f} MB)")

**Near-infrared and vegetation.** The model also sharpens the near-infrared band, which our
eyes cannot see. Plants reflect a lot of it. The false-colour image shows near-infrared as
red; NDVI turns it into a vegetation index (green = dense vegetation).

In [ ]:
LAT, LON, lr, s2 = S["lat"], S["lon"], S["lr"], S["s2"]
show([view(lr, LAT, LON, 1000), view(s2, LAT, LON, 1000)],
     ["Sentinel-2, false colour", "Stage 2, false colour"], bands=(4, 1, 2), ceiling=0.45,
     size=5.5)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (title, da) in zip(axes, [("Sentinel-2, 10 m", lr), ("Stage 2, 2.5 m", s2)]):
    im = ax.imshow(view(ndvi(da), LAT, LON, 1000).values, cmap="RdYlGn", vmin=-0.2, vmax=0.8,
                   interpolation="nearest")
    ax.set_title(f"NDVI, {title}")
    ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

**Another place, another dial.** In step 1, choose another place (or *My own place*) and
press the button again, then run the cells of steps 2 to 5 again. Or choose another
experiment in step 5 and press its button. Tips for your own place:

* pick a dry-season date (in Luzon, December to May) so there are enough clear images;
* near the coast you will get the `water` warning: the sea and reefs are not reliable;
* the area is a 2 km square around your point.

**The full walkthrough.** The notebook the presenter used,
[`demo1_super_resolution.ipynb`](https://colab.research.google.com/github/adb-sr/sr-client/blob/main/notebooks/demo1_super_resolution.ipynb),
has more: all dial settings side by side, the trade-off between sharp and truthful as a
chart, and a check against real 1 m images.

---
Results contain modified Copernicus Sentinel data. Sentinel-2 scenes are read from the public
archive on AWS (Element 84 Earth Search).